## Chapter 8 — Searching (Binary Search) · Study Notes

KTH  
July 31, 2026

---
> — "The Anatomy of A Large-Scale Hypertextual Web Search Engine,"  
> S. M. Brin and L. Page, 1998

Search algorithms can be classified in a number of ways. Is the underlying
collection **static or dynamic** (i.e., are inserts and deletes interleaved with
searching)? Is it worth spending the computational cost to **preprocess** the
data to speed up subsequent queries? Are there **statistical properties** of the
data that can be exploited? Should we operate **directly on the data or
transform it**?

This chapter focuses on **static data stored in sorted order in an array**.
(Structures appropriate for dynamic updates are covered by the Heaps, Hash
Tables, and Binary Search Trees chapters.) The first collection of problems
relates to binary search; the second pertains to general search.

## 1. Binary search

Given an **arbitrary** collection of `n` keys, the only way to determine whether
a search key is present is to examine every element — `O(n)` time.

Fundamentally, binary search is a natural **elimination-based strategy** for
searching a **sorted** array. The idea is to eliminate half the keys from
consideration at each step: if the search key is not equal to the middle element,
then one of the two sets of keys — those to the left or those to the right of the
middle — can be eliminated from further consideration.

### Why interviewers love it (and why it's dangerous)

Binary search is ideal from the interviewer's perspective: a basic technique
every reasonable candidate is supposed to know, implementable in a few lines. On
the other hand, it is **much trickier to implement correctly than it appears** —
you should implement it *and* write corner case tests to ensure you understand it
properly.

Many published implementations are incorrect in subtle and not-so-subtle ways:

- One study reported binary search was correctly implemented in only **5 out of
  20 textbooks**.
- Jon Bentley, in *Programming Pearls*, assigned binary search in a course for
  professional programmers and found **90% failed to code it correctly** despite
  ample time.
- Bentley's students would have been gratified to know that his **own published
  implementation** — in a column titled "Writing Correct Programs" — contained a
  bug that went undetected for **over twenty years**.

### 1.1 The implementation (with Bentley's bug)

Binary search can be written many ways — recursive, iterative, with different
idioms for the conditionals. Here is an iterative implementation adapted from
Bentley's book, **which includes his bug**.

In [1]:
def bsearch(t, A):
    L, U = 0, len(A) - 1
    while L <= U:
        M = (L + U) // 2        # <-- the bug lives here: L + U can overflow
        if A[M] < t:
            L = M + 1
        elif A[M] == t:
            return M
        else:
            U = M - 1
    return -1

A = [-14, -10, 2, 108, 108, 243, 285, 285, 285, 401]
print("index of 108:", bsearch(108, A))
print("index of 285:", bsearch(285, A))
print("index of 7 (absent):", bsearch(7, A))

index of 108: 4
index of 285: 7
index of 7 (absent): -1


### 1.2 The overflow bug

The error is in the midpoint assignment `M = (L + U) // 2`, which can potentially
lead to **overflow**. The overflow is avoided by computing:

```python
M = L + (U - L) // 2
```

Both expressions are mathematically equal, but the second never forms the
intermediate sum `L + U`, so it cannot exceed the larger of the two endpoints.

> 📝 **Python note.** Python's `int` is arbitrary-precision, so this particular
> overflow can't actually occur in Python — but it *is* a real, famous bug in
> languages with fixed-width integers (it was found in Java's own
> `Arrays.binarySearch` in 2006). Write it the safe way regardless: interviewers
> look for it, and the habit transfers to C/C++/Java/Go.

Here is the corrected version:

In [2]:
def bsearch_safe(t, A):
    L, U = 0, len(A) - 1
    while L <= U:
        M = L + (U - L) // 2      # overflow-safe midpoint
        if A[M] < t:
            L = M + 1
        elif A[M] == t:
            return M
        else:
            U = M - 1
    return -1

# Same results, safe arithmetic
print(bsearch_safe(108, A), bsearch_safe(285, A), bsearch_safe(7, A))

4 7 -1


In [3]:
# Demonstrating the two midpoint formulas are equivalent but differ in intermediates
L, U = 2_000_000_000, 2_100_000_000     # near the 32-bit signed limit (2_147_483_647)
print("L + U           =", L + U, "  <-- would overflow a 32-bit int")
print("(L + U) // 2    =", (L + U) // 2)
print("L + (U - L) // 2=", L + (U - L) // 2, "  <-- same answer, no large intermediate")

L + U           = 4100000000   <-- would overflow a 32-bit int
(L + U) // 2    = 2050000000
L + (U - L) // 2= 2050000000   <-- same answer, no large intermediate


### 1.3 Complexity

The time complexity is given by the recurrence `T(n) = T(n/2) + c`, where `c` is
a constant. This solves to **`T(n) = O(log n)`** — far superior to the `O(n)`
approach needed when the keys are unsorted.

**The tradeoff:** binary search requires a *sorted* array, and sorting costs
`O(n log n)`. However, if there are **many searches** to perform, the one-time
sorting cost is not an issue.

Many variants of searching a sorted array require a little more thinking and
create opportunities for **missing corner cases**.

In [4]:
# Corner case tests — the habit EPI explicitly recommends building
tests = [
    ([], 5, -1),                      # empty array
    ([5], 5, 0),                      # single element, present
    ([5], 3, -1),                     # single element, absent
    ([1, 2], 1, 0),                   # two elements, first
    ([1, 2], 2, 1),                   # two elements, second
    ([1, 3, 5], 0, -1),               # below all elements
    ([1, 3, 5], 9, -1),               # above all elements
]
for arr, target, expected in tests:
    got = bsearch_safe(target, arr)
    status = "ok " if got == expected else "FAIL"
    print(f"{status} bsearch_safe({target}, {arr}) = {got} (expected {expected})")

ok  bsearch_safe(5, []) = -1 (expected -1)
ok  bsearch_safe(5, [5]) = 0 (expected 0)
ok  bsearch_safe(3, [5]) = -1 (expected -1)
ok  bsearch_safe(1, [1, 2]) = 0 (expected 0)
ok  bsearch_safe(2, [1, 2]) = 1 (expected 1)
ok  bsearch_safe(0, [1, 3, 5]) = -1 (expected -1)
ok  bsearch_safe(9, [1, 3, 5]) = -1 (expected -1)


## 2. Searching boot camp — searching with a custom comparator

When objects are **comparable**, they can be sorted and searched using library
search functions. Languages know how to compare built-in types (integers,
strings, library classes for dates, URLs, SQL timestamps, etc.). However,
**user-defined types** used in sorted collections must explicitly implement
comparison, and must ensure that comparison has basic properties such as
**transitivity**. If comparison is implemented incorrectly, a lookup into a
sorted collection can fail *even when the item is present*.

**Setup.** Given an array of students sorted by **descending GPA**, with ties
broken on **name**, use the library binary search routine to search it quickly.
The trick is to map each student to a comparison key — `(-GPA, name)` — so that
ordinary tuple ordering reproduces "higher GPA first, then name ascending."

In [5]:
import bisect
import collections

Student = collections.namedtuple('Student', ('name', 'grade_point_average'))

def comp_gpa(student):
    return (-student.grade_point_average, student.name)

def search_student(students, target, comp_gpa):
    i = bisect.bisect_left([comp_gpa(s) for s in students], comp_gpa(target))
    return 0 <= i < len(students) and students[i] == target

In [6]:
students = [
    Student('Adam', 4.0),
    Student('Bob', 3.9),
    Student('David', 3.9),
    Student('Chris', 3.7),
    Student('Erin', 3.2),
]
# (already sorted by descending GPA, ties broken on name)

print(search_student(students, Student('David', 3.9), comp_gpa))   # True
print(search_student(students, Student('Zoe', 3.9), comp_gpa))     # False

True
False


**Complexity.** Assuming the *i*-th element of the sequence can be accessed in
`O(1)` time, the program is `O(log n)`.

> 📝 **A caveat worth noticing.** As written, the list comprehension
> `[comp_gpa(s) for s in students]` rebuilds the entire key array on *every*
> call — that's `O(n)` work, which dominates the `O(log n)` search. The `O(log n)`
> claim describes the search step itself; in practice you'd precompute the key
> array once and reuse it across many searches.

## 3. Table 8.1 — Top Tips for Searching

- **Binary search is an effective search tool.** It is applicable to more than
  just searching in sorted arrays — e.g., it can be used to search an interval
  of **real numbers or integers** (binary search on the *answer*).
- If your solution uses **sorting**, and the computation performed after sorting
  is faster than sorting itself (e.g., `O(n)` or `O(log n)`), look for solutions
  that **do not perform a complete sort**.
- Consider **time/space tradeoffs**, such as making multiple passes through the
  data.

## 4. Know your searching libraries

The **`bisect`** module provides binary search functions for a sorted `list`.
Assuming `a` is a sorted list:

| Call | Returns |
|---|---|
| `bisect.bisect_left(a, x)` | Index of the first entry **≥ `x`** (the first element *not less than* `x`). If every element is less than `x`, returns `len(a)`. |
| `bisect.bisect_right(a, x)` | Index of the first entry **> `x`** (the first element *greater than* `x`). If every element is ≤ `x`, returns `len(a)`. |

**In an interview, if it is allowed, use these functions instead of implementing
your own binary search.**

In [7]:
import bisect

a = [1, 3, 3, 3, 5, 7]

print("bisect_left(a, 3) =", bisect.bisect_left(a, 3),   "-> first index >= 3")
print("bisect_right(a, 3)=", bisect.bisect_right(a, 3),  "-> first index > 3")
print("=> 3 occurs", bisect.bisect_right(a, 3) - bisect.bisect_left(a, 3), "times")

print()
print("bisect_left(a, 4) =", bisect.bisect_left(a, 4),   "-> insertion point for 4")
print("bisect_left(a, 9) =", bisect.bisect_left(a, 9),   "-> len(a): all elements < 9")
print("bisect_right(a, 7)=", bisect.bisect_right(a, 7),  "-> len(a): all elements <= 7")
print("bisect_left(a, 0) =", bisect.bisect_left(a, 0),   "-> 0: 0 belongs at the front")

bisect_left(a, 3) = 1 -> first index >= 3
bisect_right(a, 3)= 4 -> first index > 3
=> 3 occurs 3 times

bisect_left(a, 4) = 4 -> insertion point for 4
bisect_left(a, 9) = 6 -> len(a): all elements < 9
bisect_right(a, 7)= 6 -> len(a): all elements <= 7
bisect_left(a, 0) = 0 -> 0: 0 belongs at the front


In [8]:
# Idiom: use bisect_left to build a presence test (this is problem 8.1's core)
def contains(a, x):
    i = bisect.bisect_left(a, x)
    return i < len(a) and a[i] == x       # must check bounds AND equality

print([contains(a, x) for x in (1, 2, 3, 6, 7, 8)])

[True, False, True, False, True, False]
